In [1]:
import pandas as pd
import numpy as np
import os

# dataset comes in two sheets (split by year), need to load both and combine
file_path = "../data/raw/online_retail_II.xlsx"

print("loading sheet 1...")
df1 = pd.read_excel(file_path, sheet_name="Year 2009-2010")

print("loading sheet 2...")
df2 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

df = pd.concat([df1, df2], ignore_index=True)

print(f"total rows: {df.shape[0]:,}")
print(f"columns: {df.shape[1]}")


Loading Sheet 1 (2009–2010)... this may take a minute ☕
Loading Sheet 2 (2010–2011)...

✅ Data loaded successfully!
Total rows: 1,067,371
Total columns: 8


In [2]:
# quick look before doing anything
print(df.head())

df.info()

# checking for weird values / obvious issues
print(df.describe())


=== First 5 Rows ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom



=== Column Info ===
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB

=== Basic Statistics ===


,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [3]:
rows_before = len(df)
print(f"rows before: {rows_before:,}")

# invoices starting with C are cancellations, don't want those
df = df[~df["Invoice"].astype(str).str.startswith("C")]
print(f"after removing cancellations: {len(df):,}")

# quantity and price should always be positive
rows_before = len(df)
df = df[df["Quantity"] > 0]
print(f"after removing qty <= 0: {len(df):,}")

rows_before = len(df)
df = df[df["Price"] > 0]
print(f"after removing price <= 0: {len(df):,}")


Rows before cleaning: 1,067,371
Rows after removing cancellations: 1,047,877 (removed 19,494 rows)
Rows after removing Quantity ≤ 0: 1,044,420 (removed 3,457 rows)
Rows after removing Price ≤ 0: 1,041,670 (removed 2,750 rows)

✅ Fix 1 Complete!
Final rows after Fix 1: 1,041,670


In [4]:
# customer ID is coming in as a float (17850.0) which is annoying
# converting to string since we'll never do math on it
# using Int64 with capital I because regular int64 breaks if there are nulls

print("before:")
print(df["Customer ID"].head(10))

df["Customer ID"] = df["Customer ID"].astype("Int64").astype("str")

# the nulls become the string "nan" after the conversion, fix that
df["Customer ID"] = df["Customer ID"].replace("nan", pd.NA)

print("after:")
print(df["Customer ID"].head(10))


Before fix:
0    13085.0
1    13085.0
2    13085.0
3    13085.0
4    13085.0
5    13085.0
6    13085.0
7    13085.0
8    13085.0
9    13085.0
Name: Customer ID, dtype: float64
Data type: float64

After fix:
0    13085
1    13085
2    13085
3    13085
4    13085
5    13085
6    13085
7    13085
8    13085
9    13085
Name: Customer ID, dtype: str
Data type: str


In [5]:
# some descriptions are missing - can fill them in using the stockcode
# idea: for each stockcode, find what description was used most often
# then fill the blanks with that

missing_before = df["Description"].isna().sum()
print(f"missing descriptions: {missing_before:,}")

df["Description"] = df.groupby("StockCode")["Description"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else pd.NA)
)

missing_after = df["Description"].isna().sum()
print(f"still missing: {missing_after:,}")
print(f"filled: {missing_before - missing_after:,}")


Missing descriptions before fix: 0
Missing descriptions after fix: 0
Descriptions filled: 0

✅ Fix 3 Complete!


In [6]:
dups = df.duplicated().sum()
print(f"duplicate rows: {dups:,}")

rows_before = len(df)
df = df.drop_duplicates()
print(f"dropped {rows_before - len(df):,} rows")


Duplicate rows found: 33,757
Rows before: 1,041,670
Rows after: 1,007,913
Rows removed: 33,757

✅ Fix 4 Complete!


In [7]:
df.isnull().sum()


Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    228488
Country             0
dtype: int64

In [8]:
print(f"shape: {df.shape}")
print()
print(df.isnull().sum())
print()
print(df.dtypes)
print()
display(df.head())
print(df.describe())

# sanity check - make sure nothing slipped through
print(f"rows with qty <= 0: {(df['Quantity'] <= 0).sum()}")
print(f"rows with price <= 0: {(df['Price'] <= 0).sum()}")
print(f"cancelled invoices left: {df['Invoice'].astype(str).str.startswith('C').sum()}")


=== Final DataFrame Shape ===
Rows: 1,007,913
Columns: 8

=== Missing Values Per Column ===
Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    228488
Country             0
dtype: int64

=== Data Types ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
dtype: object

=== Sample of Clean Data ===


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom



=== Basic Statistics ===


,Quantity,InvoiceDate,Price
count,1.007913e+06,1007913,1.007913e+06
mean,1.111718e+01,2011-01-04 10:34:18.759595,4.074252e+00
min,1.000000e+00,2009-12-01 07:45:00,1.000000e-03
25%,1.000000e+00,2010-07-06 11:21:00,1.250000e+00
50%,4.000000e+00,2010-12-09 15:29:00,2.100000e+00
75%,1.200000e+01,2011-07-28 12:29:00,4.130000e+00
max,8.099500e+04,2011-12-09 12:50:00,2.511109e+04
std,1.284700e+02,NaN,5.043045e+01



=== Sanity Checks ===
Rows with Quantity ≤ 0: 0
Rows with Price ≤ 0: 0
Cancelled invoices remaining: 0


In [9]:
import os

os.makedirs("../data/processed", exist_ok=True)

# saving as csv since it's way faster to load than excel next time
df.to_csv("../data/processed/online_retail_cleaned.csv", index=False)

print("saved to data/processed/online_retail_cleaned.csv")
print(f"shape: {df.shape}")


✅ Cleaned data saved to data/processed/online_retail_cleaned.csv
Final shape: 1,007,913 rows × 8 columns


In [10]:
# pulling out the date/time parts from InvoiceDate
df["Year"]       = df["InvoiceDate"].dt.year
df["Month"]      = df["InvoiceDate"].dt.month
df["Day"]        = df["InvoiceDate"].dt.day
df["DayOfWeek"]  = df["InvoiceDate"].dt.dayofweek   # 0=Mon, 6=Sun
df["Hour"]       = df["InvoiceDate"].dt.hour
df["WeekOfYear"] = df["InvoiceDate"].dt.isocalendar().week.astype(int)

# total revenue per line item - most important column for analysis
df["TotalPrice"] = df["Quantity"] * df["Price"]

print(df[["InvoiceDate", "Year", "Month", "Day", "DayOfWeek", "Hour", "WeekOfYear", "TotalPrice"]].head(10))
print(f"columns now: {df.shape[1]}")


=== New Columns Added ===
          InvoiceDate  Year  Month  Day  DayOfWeek  Hour  WeekOfYear  \
0 2009-12-01 07:45:00  2009     12    1          1     7          49   
1 2009-12-01 07:45:00  2009     12    1          1     7          49   
2 2009-12-01 07:45:00  2009     12    1          1     7          49   
3 2009-12-01 07:45:00  2009     12    1          1     7          49   
4 2009-12-01 07:45:00  2009     12    1          1     7          49   
5 2009-12-01 07:45:00  2009     12    1          1     7          49   
6 2009-12-01 07:45:00  2009     12    1          1     7          49   
7 2009-12-01 07:45:00  2009     12    1          1     7          49   
8 2009-12-01 07:46:00  2009     12    1          1     7          49   
9 2009-12-01 07:46:00  2009     12    1          1     7          49   

   TotalPrice  
0        83.4  
1        81.0  
2        81.0  
3       100.8  
4        30.0  
5        39.6  
6        30.0  
7        59.5  
8        30.6  
9        45.0  

Tota

In [11]:
df.to_csv("../data/processed/online_retail_cleaned.csv", index=False)

print(f"saved - shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
for col in df.columns:
    print(f"  {col}")


✅ Final dataset saved!
Shape: 1,007,913 rows × 15 columns

Columns saved:
  → Invoice
  → StockCode
  → Description
  → Quantity
  → InvoiceDate
  → Price
  → Customer ID
  → Country
  → Year
  → Month
  → Day
  → DayOfWeek
  → Hour
  → WeekOfYear
  → TotalPrice
